In [1]:
import json
from collections import defaultdict
from sentence_transformers import SentenceTransformer

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup 
from fastchat.model import get_conversation_template
from tqdm import tqdm
from pathlib import Path
import copy
import pickle

import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from llm2vec import LLM2Vec

from common_utils import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, roc_curve, auc, precision_recall_curve

from tqdm import tqdm
torch.manual_seed(1234)

/data/share/project/RSML/LLM_Check_Hallucination_Detection/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Projection Head Architecture

In [2]:
class ProjectionHead(nn.Module):
    def __init__(self, d_h=4096, d_m=1024, d_out=1024, max_seq_len=256, num_layers=2, nhead=8):
        super().__init__()
        # Input Projection
        self.input_proj = nn.Linear(d_h, d_m)
        
        # Learnable Positional Embeddings
        self.pos_embeddings = nn.Parameter(torch.zeros(1, max_seq_len, d_m))
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_m,
            nhead=nhead,
            dim_feedforward=4 * d_m,
            dropout=0.0,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_proj = nn.Linear(d_m, d_out)

    def forward(self, hidden_states, attention_mask):
        """
        hidden_states: (batch_size, seq_len, d_h)
        attention_mask: (batch_size, seq_len) - 1 for valid tokens, 0 for padding
        """
        seq_len = hidden_states.size(1)
        
        # Project and add positional embeddings
        x = self.input_proj(hidden_states)
        x = x + self.pos_embeddings[:, :seq_len, :]
        
        # Transform (PyTorch expects True for padded positions in key_padding_mask)
        key_padding_mask = (attention_mask == 0)
        encoded = self.transformer(x, src_key_padding_mask=key_padding_mask)
        
        # Mean Pooling over non-padding positions
        mask_expanded = attention_mask.unsqueeze(-1).expand(encoded.size()).float()
        sum_embeddings = torch.sum(encoded * mask_expanded, dim=1)
        sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / sum_mask
        
        # Output Projection and L2 Normalization
        projected = self.output_proj(pooled)
        normalized_output = F.normalize(projected, p=2, dim=-1)
        
        return normalized_output
    
class DualDistillationLoss(nn.Module):
    def __init__(self, lambda_a=0.5, lambda_c=0.5, tau=0.05):
        super().__init__()
        self.lambda_a = lambda_a
        self.lambda_c = lambda_c
        self.tau = tau

    def forward(self, student_embs, teacher_embs):
        """
        student_embs: (batch_size, 1024) - Output from your Projection Head
        teacher_embs: (batch_size, 1024) - Target from the Teacher Model
        """
        batch_size = student_embs.size(0)
        
        # 1. Alignment Loss (Cosine Distance)
        # Forces the student vector to point in the exact same direction as the teacher
        cos_sim = F.cosine_similarity(student_embs, teacher_embs, dim=-1)
        loss_align = 1.0 - cos_sim.mean()
        
        # 2. Contrastive Loss (InfoNCE)
        # Forces the student vector to be closer to ITS teacher than to other teachers in the batch
        sim_matrix = torch.matmul(student_embs, teacher_embs.T) / self.tau
        labels = torch.arange(batch_size, device=student_embs.device)
        loss_contra = F.cross_entropy(sim_matrix, labels)
            
        # Combined Objective
        total_loss = (self.lambda_a * loss_align) + (self.lambda_c * loss_contra)
                     
        return total_loss, loss_align, loss_contra

In [3]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class DualDistillationDataset(Dataset):
    def __init__(self, hidden_states_list, teacher_embs_list):
        self.hidden_states = hidden_states_list
        self.teacher_embs = teacher_embs_list

    def __len__(self):
        return len(self.hidden_states)

    def __getitem__(self, idx):
        return {
            "hidden_states": self.hidden_states[idx], # (seq_len, 4096)
            "teacher_embs": self.teacher_embs[idx]    # (1024,)
        }

def dual_collate_fn(batch):
    hidden_states = [item["hidden_states"] for item in batch]
    teacher_embs = [item["teacher_embs"] for item in batch]
    
    padded_hidden_states = pad_sequence(hidden_states, batch_first=True, padding_value=0.0)
    
    batch_size = len(hidden_states)
    max_len = padded_hidden_states.size(1)
    attention_mask = torch.zeros(batch_size, max_len, dtype=torch.long)
    
    for i, state in enumerate(hidden_states):
        attention_mask[i, :state.size(0)] = 1
        
    return {
        "hidden_states": padded_hidden_states,
        "attention_mask": attention_mask,
        "teacher_embs": torch.stack(teacher_embs) # Shape: (batch_size, 1024)
    }

In [4]:
def train_dual_distillation(hidden_states, teacher_embs, d_out=1024, epochs=80, batch_size=16):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    train_student_states, val_student_states, train_teacher_embs, val_teacher_embs = train_test_split(hidden_states, 
                                                                                                      teacher_embs, 
                                                                                                      test_size=0.2, 
                                                                                                      random_state=123)
    
    train_dataset = DualDistillationDataset(train_student_states, train_teacher_embs)
    val_dataset = DualDistillationDataset(val_student_states, val_teacher_embs)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=dual_collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=dual_collate_fn)
    
    model = ProjectionHead(d_h=4096, d_m=1024, d_out=d_out, max_seq_len=2048).to(device)
    criterion = DualDistillationLoss(lambda_a=0.5, lambda_c=0.5, tau=0.05)
    
    # Optimizer and Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=2e-4, betas=(0.9, 0.999), weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
    
    # Tracking checkpoints
    best_val_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())
    
    for epoch in range(epochs):
        # TRAINING
        model.train()
        train_loss, train_align, train_contrast = 0.0, 0.0, 0.0
        
        for batch in train_loader:
            hidden_states = batch['hidden_states'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            teacher_embs = batch['teacher_embs'].to(device)
            
            optimizer.zero_grad()
            student_embs = model(hidden_states, attention_mask)
            
            loss, l_align, l_contrast = criterion(student_embs, teacher_embs)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_align += l_align.item()
            train_contrast += l_contrast.item()
            
        scheduler.step()

        # VALIDATION
        model.eval()
        val_loss, val_align, val_contrast = 0.0, 0.0, 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                hidden_states = batch['hidden_states'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                teacher_embs = batch['teacher_embs'].to(device)
                
                student_embs = model(hidden_states, attention_mask)
                loss, l_align, l_contrast = criterion(student_embs, teacher_embs)
                
                val_loss += loss.item()
                val_align += l_align.item()
                val_contrast += l_contrast.item()
        
        train_steps = len(train_loader)
        val_steps = len(val_loader)
        
        avg_train_loss = train_loss / train_steps
        avg_val_loss = val_loss / val_steps
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            save_path = "./best_projection_head.pt"
            torch.save(best_model_weights, save_path)
            
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1:02d}/{epochs}] "
                  f"| Train Loss: {avg_train_loss:.4f} (Align: {train_align/train_steps:.4f}, Contrast: {train_contrast/train_steps:.4f}) "
                  f"| Val Loss: {avg_val_loss:.4f} (Align: {val_align/val_steps:.4f}, Contrast: {val_contrast/val_steps:.4f}) "
                  f"| LR: {scheduler.get_last_lr()[0]:.6f}")
            
    print(f"\nTraining finalized. Best Validation Loss Achieved: {best_val_loss:.4f}")

    model.load_state_dict(best_model_weights)
    return model

### Train Projection Head with External Embedding as Teacher

In [2]:
_TAGS = ["entity", "relation", "sentence", "invented", "subjective", "unverifiable"]
data_dir = Path("./data/")
model_dir = Path("../llm-hallucinations-factual-qa/.cache/models/")

def get_modified_data():
    with open("annotations.json", "r", encoding="utf-8") as f:
        data = json.loads(f.read())

    df = {
        "prompt": [],
        "output": [],
        "annotated": [],
        "modified": [],
        "model": [],
        "entity": [],
        "relation": [],
        "sentence": [],
        "invented": [],
        "subjective": [],
        "unverifiable": [],
        "hallucinated": [],
    }

    def modify(s):
        indicator = [0, 0, 0, 0, 0, 0]
        soup = BeautifulSoup(s, "html.parser")
        s1 = ""
        for t in range(len(_TAGS)):
            indicator[t] = len(soup.find_all(_TAGS[t]))
        for elem in soup.find_all(text=True):
            if elem.parent.name != "delete":
                s1 += elem
        return s1, indicator

    for i in range(len(data)):
        df["prompt"].append(data[i]["prompt"])
        df["output"].append(data[i]["output"])
        df["annotated"].append(data[i]["annotated"])
        df["model"].append(data[i]["model"])
        modified_text, indicator = modify(data[i]["annotated"])
        df["modified"].append(modified_text)
        for t in range(len(_TAGS)):
            df[_TAGS[t]].append(indicator[t])
        df["hallucinated"].append(int(sum(indicator) > 0))

    df = pd.DataFrame(df)
    return df


def get_fava_data(n_samples=200):
    np.random.seed(0)
    data = get_modified_data()
    i1 = data["hallucinated"] == 1
    i2 = data["hallucinated"] == 0
    df = pd.concat(
        [
            data[i1][:n_samples],
            data[i2][:n_samples],
        ],
        ignore_index=True,
        sort=False,
    )
    return df, []


def get_scores_dict(model_name_or_path, data, mt_list):
    system_prompt = ""
    generation_config = {}
    generation_config.update({"temperature": 0.6, "top_p": 0.9, "top_k": 50, "do_sample": True})
    model, tokenizer = load_model_and_tokenizer(model_name_or_path,
                                                dtype=torch.bfloat16,
                                                cache_dir=model_dir,
                                                attn_implementation="eager",
                                                **generation_config)
    
    print("Loading Teacher Embedding Model...")
    teacher_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='cuda' if torch.cuda.is_available() else 'cpu')
    tok_lens, labels, tok_ins = [], [], []

    scores, generated_embeddings, teacher_embeddings = [], [], []
    indiv_scores = {}
    for mt in mt_list:
        indiv_scores[mt] = defaultdict(def_dict_value)

    for i in tqdm(range(len(data))):
        row = data.loc[i]

        # define the prompt, response and labels as per the dataset
        prompt = row["prompt"]
        response = row["output"]
        labels.append(1 if row["hallucinated"] == 1 else 0)

        chat_template = get_conversation_template(model_name_or_path)
        chat_template.set_system_message(system_prompt.strip())
        chat_template.messages = []
        chat_template.append_message(chat_template.roles[0], prompt.strip())
        chat_template.append_message(chat_template.roles[1], response.strip())

        full_prompt = chat_template.get_prompt()
        user_prompt = full_prompt.split(response.strip())[0].strip()
        # print(full_prompt)

        tok_in_u = tokenizer(user_prompt, return_tensors="pt", add_special_tokens=True).input_ids
        prompt_len = tok_in_u.shape[1]
        tok_in = tokenizer(full_prompt, return_tensors="pt", add_special_tokens=True).input_ids
        # print(f"Full prompt length: {tok_in.shape[1]}")
        tok_lens.append([tok_in_u.shape[1], tok_in.shape[1]])

        logit, hidden_act, attn = get_model_vals(model, tok_in.to(1))
        last_token_embeddings = hidden_act[-1][0, prompt_len:, :].to(torch.float32).detach().cpu()  # Last token embedding from last layer
        # print(last_token_embeddings.shape)  # Should be (response_len, hidden_dim)
        generated_embeddings.append(last_token_embeddings)

        with torch.no_grad():
            # Distilling the mapping of the generated text output into the embedding target
            t_emb = teacher_model.encode(response, convert_to_tensor=True).cpu().float()
            teacher_embeddings.append(t_emb)

        # Unpacking the values into lists on CPU
        logit = logit[0].cpu()
        hidden_act = [x[0].to(torch.float32).detach().cpu() for x in hidden_act]
        attn = [x[0].to(torch.float32).detach().cpu() for x in attn]
        tok_in = tok_in.cpu()

        tok_len = [tok_in_u.shape[1], tok_in.shape[1]]
        compute_scores(
            [logit],
            [hidden_act],
            [attn],
            scores,
            indiv_scores,
            mt_list,
            [tok_in],
            [tok_len],
            # use_toklens=args.use_toklens,
        )

    return scores, indiv_scores, generated_embeddings, teacher_embeddings, labels

In [ ]:
with open('data/projection_head/scores_fava_annot_llama3_layer32_500samp.pkl', 'rb') as f:
    scores, sample_indiv_scores, generated_embeddings, teacher_embeddings, sample_labels = pickle.load(f)

In [ ]:
trained_model = train_dual_distillation(generated_embeddings, teacher_embeddings, d_out=1024, epochs=50, batch_size=16)

Epoch [01/50] | Train Loss: 0.8964 (Align: 0.6302, Contrast: 1.1626) | Val Loss: 0.4016 (Align: 0.4666, Contrast: 0.3365) | LR: 0.000200
Epoch [05/50] | Train Loss: 0.1053 (Align: 0.1389, Contrast: 0.0717) | Val Loss: 0.1293 (Align: 0.1702, Contrast: 0.0884) | LR: 0.000195
Epoch [10/50] | Train Loss: 0.0589 (Align: 0.0638, Contrast: 0.0540) | Val Loss: 0.0953 (Align: 0.1133, Contrast: 0.0773) | LR: 0.000182
Epoch [15/50] | Train Loss: 0.0243 (Align: 0.0293, Contrast: 0.0193) | Val Loss: 0.0820 (Align: 0.0874, Contrast: 0.0766) | LR: 0.000161
Epoch [20/50] | Train Loss: 0.0244 (Align: 0.0175, Contrast: 0.0314) | Val Loss: 0.0757 (Align: 0.0745, Contrast: 0.0770) | LR: 0.000134
Epoch [25/50] | Train Loss: 0.0215 (Align: 0.0121, Contrast: 0.0310) | Val Loss: 0.0750 (Align: 0.0717, Contrast: 0.0783) | LR: 0.000105
Epoch [30/50] | Train Loss: 0.0249 (Align: 0.0124, Contrast: 0.0374) | Val Loss: 0.0736 (Align: 0.0697, Contrast: 0.0776) | LR: 0.000076
Epoch [35/50] | Train Loss: 0.0189 (Align

### Train projection head with LLM2Vec

In [ ]:
_TAGS = ["entity", "relation", "sentence", "invented", "subjective", "unverifiable"]
data_dir = Path("./data/")
model_dir = Path("../llm-hallucinations-factual-qa/.cache/models/")

def get_modified_data():
    with open("annotations.json", "r", encoding="utf-8") as f:
        data = json.loads(f.read())

    df = {
        "prompt": [],
        "output": [],
        "annotated": [],
        "modified": [],
        "model": [],
        "entity": [],
        "relation": [],
        "sentence": [],
        "invented": [],
        "subjective": [],
        "unverifiable": [],
        "hallucinated": [],
    }

    def modify(s):
        indicator = [0, 0, 0, 0, 0, 0]
        soup = BeautifulSoup(s, "html.parser")
        s1 = ""
        for t in range(len(_TAGS)):
            indicator[t] = len(soup.find_all(_TAGS[t]))
        for elem in soup.find_all(text=True):
            if elem.parent.name != "delete":
                s1 += elem
        return s1, indicator

    for i in range(len(data)):
        df["prompt"].append(data[i]["prompt"])
        df["output"].append(data[i]["output"])
        df["annotated"].append(data[i]["annotated"])
        df["model"].append(data[i]["model"])
        modified_text, indicator = modify(data[i]["annotated"])
        df["modified"].append(modified_text)
        for t in range(len(_TAGS)):
            df[_TAGS[t]].append(indicator[t])
        df["hallucinated"].append(int(sum(indicator) > 0))

    df = pd.DataFrame(df)
    return df


def get_fava_data(n_samples=200):
    np.random.seed(0)
    data = get_modified_data()
    i1 = data["hallucinated"] == 1
    i2 = data["hallucinated"] == 0
    df = pd.concat(
        [
            data[i1][:n_samples],
            data[i2][:n_samples],
        ],
        ignore_index=True,
        sort=False,
    )
    return df, []


def get_scores_dict(model_name_or_path, data, mt_list):
    system_prompt = ""
    generation_config = {}
    generation_config.update({"temperature": 0.6, "top_p": 0.9, "top_k": 50, "do_sample": True})
    model, tokenizer = load_model_and_tokenizer(model_name_or_path,
                                                dtype=torch.bfloat16,
                                                cache_dir=model_dir,
                                                attn_implementation="eager",
                                                **generation_config)
    # tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, cache_dir=model_dir,)
    # tokenizer.padding_side = "left"
    # if tokenizer.pad_token is None:
    #     tokenizer.pad_token = "[PAD]"
    
    print("Loading Teacher LLM2Vec Model...")
    teacher_model = LLM2Vec.from_pretrained(
        "meta-llama/Meta-Llama-3-8B-Instruct",
        peft_model_name_or_path="McGill-NLP/LLM2Vec-Meta-Llama-3-8B-Instruct-mntp",
        cache_dir=model_dir,
        device_map='cuda:1' if torch.cuda.is_available() else 'cpu',
        torch_dtype=torch.bfloat16,
    )
    # with open('data/projection_head/scores_fava_annot_llama3_layer32_500samp.pkl', 'rb') as f:
    #     scores, indiv_scores, generated_embeddings, _, labels = pickle.load(f)
    tok_lens, labels, tok_ins = [], [], []
    scores, generated_embeddings, teacher_embeddings = [], [], []
    indiv_scores = {}
    for mt in mt_list:
        indiv_scores[mt] = defaultdict(def_dict_value)

    for i in tqdm(range(len(data))):
        row = data.loc[i]

        # define the prompt, response and labels as per the dataset
        prompt = row["prompt"]
        response = row["output"]
        labels.append(1 if row["hallucinated"] == 1 else 0)

        chat_template = get_conversation_template(model_name_or_path)
        chat_template.set_system_message(system_prompt.strip())
        chat_template.messages = []
        chat_template.append_message(chat_template.roles[0], prompt.strip())
        chat_template.append_message(chat_template.roles[1], response.strip())

        full_prompt = chat_template.get_prompt()
        user_prompt = full_prompt.split(response.strip())[0].strip()
        # print(full_prompt)

        tok_in_u = tokenizer(user_prompt, return_tensors="pt", add_special_tokens=True).input_ids
        prompt_len = tok_in_u.shape[1]
        tok_in = tokenizer(full_prompt, return_tensors="pt", add_special_tokens=True).input_ids
        # print(f"Full prompt length: {tok_in.shape[1]}")
        tok_lens.append([tok_in_u.shape[1], tok_in.shape[1]])

        logit, hidden_act, attn = get_model_vals(model, tok_in.to(1))
        last_token_embeddings = hidden_act[-1][0, prompt_len:, :].to(torch.float32).detach().cpu()  # Last token embedding from last layer
        # print(last_token_embeddings.shape)  # Should be (response_len, hidden_dim)
        generated_embeddings.append(last_token_embeddings)

        with torch.no_grad():
            t_emb = teacher_model.encode([response])[0].cpu().float()
            teacher_embeddings.append(t_emb)

        # Unpacking the values into lists on CPU
        logit = logit[0].cpu()
        hidden_act = [x[0].to(torch.float32).detach().cpu() for x in hidden_act]
        attn = [x[0].to(torch.float32).detach().cpu() for x in attn]
        tok_in = tok_in.cpu()

        tok_len = [tok_in_u.shape[1], tok_in.shape[1]]
        compute_scores(
            [logit],
            [hidden_act],
            [attn],
            scores,
            indiv_scores,
            mt_list,
            [tok_in],
            [tok_len],
            # use_toklens=args.use_toklens,
        )

    return scores, indiv_scores, generated_embeddings, teacher_embeddings, labels

In [4]:
model_name_or_path = get_full_model_name("llama3")[1]
n_samples = 500
mt_list = ["logit", "attns"]

sample_data, _ = get_fava_data(n_samples=n_samples)

# get scores for sample data
scores, sample_indiv_scores, generated_embeddings, teacher_embeddings, sample_labels = get_scores_dict(model_name_or_path, sample_data, mt_list)
with open(f"data/projection_head_llm2vec/scores_fava_annot_llama3_layer32_{n_samples}samp.pkl", "wb") as f:
    pickle.dump([scores, sample_indiv_scores, generated_embeddings, teacher_embeddings, sample_labels], f)

/tmp/ipykernel_1106459/2119670236.py:30: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  for elem in soup.find_all(text=True):


Loading Teacher LLM2Vec Model...


100%|██████████| 460/460 [44:14<00:00,  5.77s/it]


In [5]:
with open('data/projection_head_llm2vec/scores_fava_annot_llama3_layer32_500samp.pkl', 'rb') as f:
    scores, sample_indiv_scores, generated_embeddings, teacher_embeddings, sample_labels = pickle.load(f)

In [6]:
trained_model = train_dual_distillation(generated_embeddings, teacher_embeddings, d_out=4096, epochs=100, batch_size=32)

/data/share/project/RSML/LLM_Check_Hallucination_Detection/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Epoch [01/100] | Train Loss: 28.6506 (Align: 1.0048, Contrast: 56.2964) | Val Loss: 20.8521 (Align: 1.0059, Contrast: 40.6984) | LR: 0.000200
Epoch [05/100] | Train Loss: 1.6457 (Align: 0.8353, Contrast: 2.4562) | Val Loss: 1.9225 (Align: 0.8425, Contrast: 3.0024) | LR: 0.000199
Epoch [10/100] | Train Loss: 1.2490 (Align: 0.7953, Contrast: 1.7026) | Val Loss: 1.4594 (Align: 0.7752, Contrast: 2.1437) | LR: 0.000195
Epoch [15/100] | Train Loss: 1.0879 (Align: 0.6789, Contrast: 1.4968) | Val Loss: 1.1246 (Align: 0.6787, Contrast: 1.5705) | LR: 0.000190
Epoch [20/100] | Train Loss: 0.9157 (Align: 0.5901, Contrast: 1.2412) | Val Loss: 1.2252 (Align: 0.5988, Contrast: 1.8515) | LR: 0.000182
Epoch [25/100] | Train Loss: 0.7175 (Align: 0.4524, Contrast: 0.9827) | Val Loss: 0.7077 (Align: 0.4463, Contrast: 0.9692) | LR: 0.000172
Epoch [30/100] | Train Loss: 0.5722 (Align: 0.3631, Contrast: 0.7813) | Val Loss: 1.0232 (Align: 0.3720, Contrast: 1.6743) | LR: 0.000161
Epoch [35/100] | Train Loss: 0

### Train classifier

In [7]:
batch_size = 16
dropout_mlp = 0.5
dropout_gru = 0.25
learning_rate = 1e-4
weight_decay = 1e-2
gpu = "1"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda:1


In [8]:
def build_attention_mask(hidden_states):
    padded_hidden_states = pad_sequence(hidden_states, batch_first=True, padding_value=0.0)
    batch_size = len(hidden_states)
    max_len = padded_hidden_states.size(1)
    attention_mask = torch.zeros(batch_size, max_len, dtype=torch.long)
    
    for i, state in enumerate(hidden_states):
        attention_mask[i, :state.size(0)] = 1

    return attention_mask

In [9]:
with open('data/projection_head_llm2vec/scores_fava_annot_llama3_layer32_500samp.pkl', 'rb') as f:
    scores, sample_indiv_scores, generated_embeddings, teacher_embeddings, sample_labels = pickle.load(f)
    
attention_masks = []
for idx in range(len(generated_embeddings)):
    attention_mask = build_attention_mask([generated_embeddings[idx]])
    attention_masks.append(attention_mask)

projection_head = ProjectionHead(d_h=4096, d_m=1024, d_out=4096, max_seq_len=2048).to(device)
projection_head.load_state_dict(torch.load('./best_projection_head.pt', map_location=device))

<All keys matched successfully>

In [10]:
proj_head_outputs = []
projection_head.eval()
for idx in range(len(generated_embeddings)):
    generated_embedding = torch.tensor(generated_embeddings[idx]).to(device).view(1, -1, 4096)
    attention_mask = torch.tensor(attention_masks[idx]).unsqueeze(0).to(device).view(1, -1)
    proj_head_output = projection_head(generated_embedding, attention_mask)
    proj_head_output = proj_head_output.detach().squeeze().cpu().numpy()
    proj_head_outputs.append(proj_head_output)

proj_head_outputs = np.stack(proj_head_outputs)

/tmp/ipykernel_1191709/1507911435.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  generated_embedding = torch.tensor(generated_embeddings[idx]).to(device).view(1, -1, 4096)
/tmp/ipykernel_1191709/1507911435.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(attention_masks[idx]).unsqueeze(0).to(device).view(1, -1)


In [11]:
class FFHallucinationClassifier(torch.nn.Module):
    def __init__(self, input_shape, dropout = dropout_mlp):
        super().__init__()
        self.dropout = dropout
        
        self.linear_relu_stack = torch.nn.Sequential(
            torch.nn.Linear(input_shape, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(self.dropout),
            torch.nn.Linear(256, 2)
            )

    def forward(self, x):
        logits = self.linear_relu_stack(x)
        return logits

def gen_classifier_roc(inputs, labels):
    X_train, X_test, y_train, y_test = train_test_split(inputs, labels, test_size=0.2, random_state=123)
    classifier_model = FFHallucinationClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        y_score = pred[:,1].cpu().numpy()
        y_true = y_test.cpu().numpy()
        roc_auc = roc_auc_score(y_true, y_score)
        acc = (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
        f1 = f1_score(y_true, prediction_classes)
        fpr, tpr, thresholds = roc_curve(y_true, y_score)
        tpr_at_5_fpr = np.interp(0.05, fpr, tpr)
    return roc_auc, acc, tpr_at_5_fpr, f1

In [12]:
context_emb_roc, context_emb_acc, context_emb_tpr_at_5_fpr, context_emb_f1 = gen_classifier_roc(proj_head_outputs, sample_labels)

100%|██████████| 1001/1001 [00:00<00:00, 1030.13it/s]


In [13]:
print(f"Contextual Embeddings - AUROC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")
print(f"f1_score: {context_emb_f1:.4f}")
print(f"TPR at 5% FPR: {context_emb_tpr_at_5_fpr:.4f}")

Contextual Embeddings - AUROC: 0.6284, Accuracy: 0.6957
f1_score: 0.7971
TPR at 5% FPR: 0.1746


In [14]:
class ResidualBlock(torch.nn.Module):
    def __init__(self, hidden_dim):
        super(ResidualBlock, self).__init__()
        self.fc1 = torch.nn.Linear(hidden_dim, hidden_dim)
        self.relu = torch.nn.ReLU()
        # self.fc2 = torch.nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        identity = x
        
        out = self.fc1(x)
        # out = self.relu(out)
        # out = self.fc2(out)
        
        out = out + identity
        out = self.relu(out)
        
        return out

class DNNClassifier(torch.nn.Module):
    def __init__(self, input_size, hidden_size=256):
        super(DNNClassifier, self).__init__()
        
        self.layer1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()

        self.res_block = ResidualBlock(hidden_size)

        self.layer3 = torch.nn.Linear(hidden_size, 2)
        
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        
        x = self.res_block(x)

        x = self.layer3(x)
        
        return x

def gen_classifier_roc(inputs, labels):
    X_train, X_test, y_train, y_test = train_test_split(inputs, labels, test_size=0.2, random_state=123)
    classifier_model = DNNClassifier(X_train.shape[1]).to(device)
    X_train = torch.tensor(X_train).to(device)
    y_train = torch.tensor(y_train).to(torch.long).to(device)
    X_test = torch.tensor(X_test).to(device)
    y_test = torch.tensor(y_test).to(torch.long).to(device)

    optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    for _ in tqdm(range(1001)):
        optimizer.zero_grad()
        sample = torch.randperm(X_train.shape[0])[:batch_size]
        pred = classifier_model(X_train[sample])
        loss = torch.nn.functional.cross_entropy(pred, y_train[sample])
        loss.backward()
        optimizer.step()
    classifier_model.eval()
    with torch.no_grad():
        pred = torch.nn.functional.softmax(classifier_model(X_test), dim=1)
        prediction_classes = (pred[:,1]>0.5).type(torch.long).cpu()
        y_score = pred[:,1].cpu().numpy()
        y_true = y_test.cpu().numpy()
        roc_auc = roc_auc_score(y_true, y_score)
        acc = (prediction_classes.numpy()==y_test.cpu().numpy()).mean()
        f1 = f1_score(y_true, prediction_classes)
        fpr, tpr, thresholds = roc_curve(y_true, y_score)
        tpr_at_5_fpr = np.interp(0.05, fpr, tpr)
    return roc_auc, acc, tpr_at_5_fpr, f1

In [15]:
context_emb_roc, context_emb_acc, context_emb_tpr_at_5_fpr, context_emb_f1 = gen_classifier_roc(proj_head_outputs, sample_labels)

100%|██████████| 1001/1001 [00:03<00:00, 311.69it/s]


In [16]:
print(f"Contextual Embeddings - AUROC: {context_emb_roc:.4f}, Accuracy: {context_emb_acc:.4f}")
print(f"f1_score: {context_emb_f1:.4f}")
print(f"TPR at 5% FPR: {context_emb_tpr_at_5_fpr:.4f}")

Contextual Embeddings - AUROC: 0.6552, Accuracy: 0.6630
f1_score: 0.7737
TPR at 5% FPR: 0.3810
